In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches


## Load DA-level Census Data

In [ ]:
DA_Census = gpd.read_file("data/census/DA/Toronto_DAs_.gpkg")
print(f"Loaded {len(DA_Census):,} dissemination areas")
print(f"CRS: {DA_Census.crs}")


## Convert Census Columns to Numeric

In [ ]:
def safe_float(col):
    """Convert a census column to float, handling suppressed/missing values."""
    return pd.to_numeric(
        col.astype(str)
        .str.replace(',', '', regex=False)
        .str.strip()
        .replace({'': '0', 'nan': '0', 'NaN': '0', '...': '0', 'x': '0'}),
        errors='coerce'
    ).fillna(0)

# Column names
col_pop          = 'Population and dwelling counts / Population, 2021'
col_unemployed   = 'Labour - Total Sex / Total - Population aged 15 years and over by labour force status - 25% sample data ; Both sexes / In the labour force ; Both sexes / Unemployed ; Both sexes '
col_labour_force = 'Labour - Total Sex / Total - Population aged 15 years and over by labour force status - 25% sample data ; Both sexes / In the labour force ; Both sexes'
col_core_need    = 'Housing - Total Sex / Total - Owner and tenant households with household total income greater than zero and shelter-cost-to-income ratio less than 100%, in non-farm, non-reserve private dwellings - 25% sample data / In core need '
col_housing_tot  = 'Housing - Total Sex / Total - Owner and tenant households with household total income greater than zero and shelter-cost-to-income ratio less than 100%, in non-farm, non-reserve private dwellings - 25% sample data '
col_first_gen    = 'Immigration - Total Sex / Total - Generation status for the population in private households - 25% sample data ; Both sexes / First generation ; Both sexes '
col_gen_total    = 'Immigration - Total Sex / Total - Generation status for the population in private households - 25% sample data ; Both sexes '
col_recent_imm   = 'Immigration - Total Sex / Total - Immigrant status and period of immigration for the population in private households - 25% sample data ; Both sexes / Immigrants ; Both sexes / 2011 to 2021 ; Both sexes / 2016 to 2021 ; Both sexes'
col_imm_total    = 'Immigration - Total Sex / Total - Immigrant status and period of immigration for the population in private households - 25% sample data ; Both sexes / Immigrants ; Both sexes'
col_vm           = ' Indigenous and Visible Minorities - Total Sex / Total - Visible minority for the population in private households - 25% sample data ; Both sexes / Total visible minority population ; Both sexes '
col_vm_total     = 'Indigenous and Visible Minorities - Total Sex / Total - Visible minority for the population in private households - 25% sample data ; Both sexes '
col_li_1         = 'Income - Total Sex / Total - After-tax income groups in 2020 for the population aged 15 years and over in private households - 100% data ; Both sexes / With after-tax income ; Both sexes / Under $10,000 (including loss) ; Both sexes'
col_li_2         = 'Income - Total Sex / Total - After-tax income groups in 2020 for the population aged 15 years and over in private households - 100% data ; Both sexes / With after-tax income ; Both sexes / $10,000 to $19,999 ; Both sexes '
col_li_3         = 'Income - Total Sex / Total - After-tax income groups in 2020 for the population aged 15 years and over in private households - 100% data ; Both sexes / With after-tax income ; Both sexes / $20,000 to $29,999 ; Both sexes'
col_seniors      = 'Age & Sex - Both sexes / Total - Age groups of the population - 100% data ; Both sexes / 65 years and over ; Both sexes'
col_children     = 'Age & Sex - Both sexes / Total - Age groups of the population - 100% data ; Both sexes / 0 to 14 years ; Both sexes'

for col in [col_pop, col_unemployed, col_labour_force, col_core_need, col_housing_tot,
            col_first_gen, col_gen_total, col_recent_imm, col_imm_total,
            col_vm, col_vm_total, col_li_1, col_li_2, col_li_3,
            col_seniors, col_children]:
    if col in DA_Census.columns:
        DA_Census[col] = safe_float(DA_Census[col])
    else:
        print(f"WARNING: column not found — {col[:60]}...")
        DA_Census[col] = 0

print("✓ Columns converted to numeric")


## Calculate Demographic Percentages

In [ ]:
pop = DA_Census[col_pop].replace(0, np.nan)

DA_Census['Unemployed, %']                    = DA_Census[col_unemployed] / DA_Census[col_labour_force] * 100
DA_Census['In core housing need, %']          = DA_Census[col_core_need]  / DA_Census[col_housing_tot]  * 100
DA_Census['First generation immigrants, %']   = DA_Census[col_first_gen]  / DA_Census[col_gen_total]    * 100
DA_Census['Immigrated within the last 10 years, %'] = DA_Census[col_recent_imm] / DA_Census[col_imm_total] * 100
DA_Census['Visible minority, %']              = DA_Census[col_vm]         / DA_Census[col_vm_total]     * 100
DA_Census['Seniors, %']                       = DA_Census[col_seniors]    / pop                         * 100
DA_Census['Children & Pre-teens, %']          = DA_Census[col_children]   / pop                         * 100

pct_cols = ['Unemployed, %', 'In core housing need, %', 'First generation immigrants, %',
            'Immigrated within the last 10 years, %', 'Visible minority, %',
            'Seniors, %', 'Children & Pre-teens, %']

for col in pct_cols:
    DA_Census[col] = DA_Census[col].fillna(0)

print("✓ Percentages calculated")
print(DA_Census[pct_cols].describe().round(1))


## Aggregate DA Demographics to Census Tracts

Generates `census_tracts_demographics.geojson` — CT-level demographic percentages for the choropleth maps.

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np

# ── STEP 1: Load DA Census data ───────────────────────────────────────────
print("Loading DA Census data...")
DA_Census = gpd.read_file("data/census/DA/Toronto_DAs_.gpkg")
print(f"Loaded {len(DA_Census)} DAs")

# ── STEP 2: Safe numeric converter ────────────────────────────────────────
def safe_float(col):
    return pd.to_numeric(
        col.astype(str)
        .str.replace(',', '', regex=False)
        .str.strip()
        .replace({'': '0', 'nan': '0', 'NaN': '0', '...': '0', 'x': '0'}),
        errors='coerce'
    ).fillna(0)

# ── STEP 3: Column names ──────────────────────────────────────────────────
col_unemployed   = 'Labour - Total Sex / Total - Population aged 15 years and over by labour force status - 25% sample data ; Both sexes / In the labour force ; Both sexes / Unemployed ; Both sexes '
col_labour_force = 'Labour - Total Sex / Total - Population aged 15 years and over by labour force status - 25% sample data ; Both sexes / In the labour force ; Both sexes'
col_core_need    = 'Housing - Total Sex / Total - Owner and tenant households with household total income greater than zero and shelter-cost-to-income ratio less than 100%, in non-farm, non-reserve private dwellings - 25% sample data / In core need '
col_housing_tot  = 'Housing - Total Sex / Total - Owner and tenant households with household total income greater than zero and shelter-cost-to-income ratio less than 100%, in non-farm, non-reserve private dwellings - 25% sample data '
col_first_gen    = 'Immigration - Total Sex / Total - Generation status for the population in private households - 25% sample data ; Both sexes / First generation ; Both sexes '
col_gen_total    = 'Immigration - Total Sex / Total - Generation status for the population in private households - 25% sample data ; Both sexes '
col_recent_imm   = 'Immigration - Total Sex / Total - Immigrant status and period of immigration for the population in private households - 25% sample data ; Both sexes / Immigrants ; Both sexes / 2011 to 2021 ; Both sexes / 2016 to 2021 ; Both sexes'
col_imm_total    = 'Immigration - Total Sex / Total - Immigrant status and period of immigration for the population in private households - 25% sample data ; Both sexes / Immigrants ; Both sexes'
col_vm           = ' Indigenous and Visible Minorities - Total Sex / Total - Visible minority for the population in private households - 25% sample data ; Both sexes / Total visible minority population ; Both sexes '
col_vm_total     = 'Indigenous and Visible Minorities - Total Sex / Total - Visible minority for the population in private households - 25% sample data ; Both sexes '
col_li_1         = 'Income - Total Sex / Total - After-tax income groups in 2020 for the population aged 15 years and over in private households - 100% data ; Both sexes / With after-tax income ; Both sexes / Under $10,000 (including loss) ; Both sexes'
col_li_2         = 'Income - Total Sex / Total - After-tax income groups in 2020 for the population aged 15 years and over in private households - 100% data ; Both sexes / With after-tax income ; Both sexes / $10,000 to $19,999 ; Both sexes '
col_li_3         = 'Income - Total Sex / Total - After-tax income groups in 2020 for the population aged 15 years and over in private households - 100% data ; Both sexes / With after-tax income ; Both sexes / $20,000 to $29,999 ; Both sexes'
col_seniors      = 'Age & Sex - Both sexes / Total - Age groups of the population - 100% data ; Both sexes / 65 years and over ; Both sexes'
col_children     = 'Age & Sex - Both sexes / Total - Age groups of the population - 100% data ; Both sexes / 0 to 14 years ; Both sexes'
col_pop          = 'Population and dwelling counts / Population, 2021'

# ── STEP 4: Convert to numeric ────────────────────────────────────────────
print("Converting columns to numeric...")
for col in [col_unemployed, col_labour_force, col_core_need, col_housing_tot,
            col_first_gen, col_gen_total, col_recent_imm, col_imm_total,
            col_vm, col_vm_total, col_li_1, col_li_2, col_li_3,
            col_seniors, col_children, col_pop]:
    if col in DA_Census.columns:
        DA_Census[col] = safe_float(DA_Census[col])
    else:
        print(f"  WARNING: not found — {col[:60]}...")
        DA_Census[col] = 0

# ── STEP 5: Calculate counts ──────────────────────────────────────────────
print("Calculating counts...")

DA_Census['Unemployed_count']          = DA_Census[col_unemployed].fillna(0)
DA_Census['Core_housing_need_count']   = DA_Census[col_core_need].fillna(0)
DA_Census['Recent_immigrant_count']    = DA_Census[col_recent_imm].fillna(0)
DA_Census['First_gen_immigrant_count'] = DA_Census[col_first_gen].fillna(0)
DA_Census['Visible_minority_count']    = DA_Census[col_vm].fillna(0)
DA_Census['Low_income_count']          = (
    DA_Census[col_li_1].fillna(0) +
    DA_Census[col_li_2].fillna(0) +
    DA_Census[col_li_3].fillna(0)
)
DA_Census['Seniors_count']             = DA_Census[col_seniors].fillna(0)
DA_Census['Children_count']            = DA_Census[col_children].fillna(0)
DA_Census['pop_total']                 = DA_Census[col_pop].fillna(0)

print("✓ Counts calculated")

# ── STEP 6: Load CT boundaries clipped to Toronto ─────────────────────────
print("Loading CT boundaries...")
toronto_bound = gpd.read_file("data/toronto-boundary-wgs84(1)/citygcs_regional_mun_wgs84.shp")
toronto_bound = toronto_bound.to_crs("EPSG:4326")

ct = gpd.read_file("data/census/CT/CT_bound/lct_000a21a_e.shp")[['CTUID', 'geometry']]
ct = ct.to_crs("EPSG:4326")
ct = gpd.clip(ct, toronto_bound)
print(f"✓ Loaded {len(ct)} CT polygons within Toronto")

# ── STEP 7: Spatial join DA → CT ──────────────────────────────────────────
print("Spatially joining DAs to CTs...")
da_wgs = DA_Census.to_crs("EPSG:4326")
da_centroids = da_wgs.copy()
da_centroids['geometry'] = da_wgs.to_crs("EPSG:26917").geometry.centroid.to_crs("EPSG:4326")

da_to_ct = gpd.sjoin(
    da_centroids[['DAUID', 'geometry']],
    ct[['CTUID', 'geometry']],
    how='left',
    predicate='within'
)

print(f"✓ Matched {da_to_ct['CTUID'].notna().sum()} of {len(da_to_ct)} DAs to CTs")

# Merge CTUID back into DA_Census
DA_Census['CTUID'] = da_to_ct['CTUID'].values

# ── STEP 8: Aggregate DA → CT ─────────────────────────────────────────────
print("Aggregating to CT level...")

count_cols = ['pop_total', 'Unemployed_count', 'Core_housing_need_count',
              'Recent_immigrant_count', 'First_gen_immigrant_count',
              'Visible_minority_count', 'Low_income_count',
              'Seniors_count', 'Children_count']

ct_agg = DA_Census.groupby('CTUID')[count_cols].sum().reset_index()
ct_agg = ct_agg.rename(columns={
    'Unemployed_count':            'unemployed',
    'Core_housing_need_count':     'core_housing_need',
    'Recent_immigrant_count':      'recent_immigrants',
    'First_gen_immigrant_count':   'first_gen_immigrants',
    'Visible_minority_count':      'visible_minority',
    'Low_income_count':            'low_income',
    'Seniors_count':               'seniors',
    'Children_count':              'children',
})

print(f"✓ Aggregated to {len(ct_agg)} census tracts")

# ── STEP 9: Merge with CT geometries ──────────────────────────────────────
print("Merging with CT geometries...")
ct_final = ct.merge(ct_agg, on='CTUID', how='left')

fill_cols = ['pop_total', 'unemployed', 'core_housing_need', 'recent_immigrants',
             'first_gen_immigrants', 'visible_minority', 'low_income', 'seniors', 'children']
ct_final[fill_cols] = ct_final[fill_cols].fillna(0)

# Remove non-residential tracts
ct_final = ct_final[ct_final['pop_total'] > 0].copy()

# Calculate percentages, capped at 100
for col in ['visible_minority', 'low_income', 'recent_immigrants',
            'first_gen_immigrants', 'seniors', 'children']:
    ct_final[f'{col}_pct'] = (
        (ct_final[col] / ct_final['pop_total']) * 100
    ).clip(upper=100)

# ── STEP 10: Save ─────────────────────────────────────────────────────────
ct_final.to_file("census_tracts_demographics.geojson", driver='GeoJSON')
print(f"\n✓ Saved census_tracts_demographics.geojson — {len(ct_final)} tracts")
print(ct_final[['visible_minority_pct','low_income_pct',
                'seniors_pct','recent_immigrants_pct']].describe().round(1))

# ── STEP 11: Quantile breaks for SwipeMap.svelte ──────────────────────────
print("\nQuantile breaks for SwipeMap.svelte:")
for col in ['visible_minority_pct', 'low_income_pct', 'recent_immigrants_pct',
            'first_gen_immigrants_pct', 'seniors_pct', 'children_pct']:
    q = ct_final[col].quantile([0.2, 0.4, 0.6, 0.8]).round(1).tolist()
    print(f"  {col}: {q}")

## Weighted Average Travel Time by Demographic Group

Computes population-weighted average travel times for walk-only and walk+transit scenarios.

In [ ]:
# ============================================================================
# LIBRARY ACCESS ANALYSIS: Walking vs Walking+Transit Comparison (FIXED)
# ============================================================================

import pandas as pd
import geopandas as gpd
import numpy as np

# ----------------------------------------------------------------------------
# PART 1: Load and process BOTH travel time datasets
# ----------------------------------------------------------------------------
print("="*65)
print("STEP 1: Loading travel time data for both modes")
print("="*65)

# Load hexagon shapes
hexagons = gpd.read_file("data/toronto_hexgrid_res9_clipped_by_bound.geojson")
hexagons['hex_id'] = hexagons.index.astype(int)  # Ensure integer type

print(f"Loaded {len(hexagons):,} hexagons")

# ----- Mode 1: Walking + Transit -----
print("\n1. Loading Walking+Transit travel times...")
tt_walk_transit = pd.read_csv("library_travel_times_both.csv", 
                              skiprows=1, 
                              names=['from_id', 'to_id', 'travel_time'])

tt_walk_transit['travel_time'] = pd.to_numeric(tt_walk_transit['travel_time'], errors='coerce')
tt_walk_transit.loc[tt_walk_transit['travel_time'] > 120, 'travel_time'] = 120
tt_walk_transit_clean = tt_walk_transit.dropna(subset=['travel_time'])

min_walk_transit = tt_walk_transit_clean.groupby('from_id')['travel_time'].min().reset_index()
min_walk_transit.columns = ['hex_id', 'walk_transit_minutes']
min_walk_transit['hex_id'] = min_walk_transit['hex_id'].astype(int)  # Ensure integer type

# ----- Mode 2: Walking Only -----
print("2. Loading Walking-Only travel times...")

try:
    tt_walk_only = pd.read_csv("library_travel_times_walking.csv", 
                               skiprows=1, 
                               names=['from_id', 'to_id', 'travel_time'])
    tt_walk_only['travel_time'] = pd.to_numeric(tt_walk_only['travel_time'], errors='coerce')
    tt_walk_only.loc[tt_walk_only['travel_time'] > 120, 'travel_time'] = 120
    tt_walk_only_clean = tt_walk_only.dropna(subset=['travel_time'])
    
    min_walk_only = tt_walk_only_clean.groupby('from_id')['travel_time'].min().reset_index()
    min_walk_only.columns = ['hex_id', 'walk_only_minutes']
    min_walk_only['hex_id'] = min_walk_only['hex_id'].astype(int)  # Ensure integer type
    
    # Merge both modes with hexagons
    hex_with_times = hexagons.merge(min_walk_transit, on='hex_id', how='left')
    hex_with_times = hex_with_times.merge(min_walk_only, on='hex_id', how='left')
    
    # Fill missing values
    hex_with_times['walk_transit_minutes'] = hex_with_times['walk_transit_minutes'].fillna(120)
    hex_with_times['walk_only_minutes'] = hex_with_times['walk_only_minutes'].fillna(120)
    
    print(f"✓ Loaded both modes for {len(hex_with_times):,} hexagons")
    print(f"  Walking+Transit - Mean: {hex_with_times['walk_transit_minutes'].mean():.1f} min")
    print(f"  Walking Only     - Mean: {hex_with_times['walk_only_minutes'].mean():.1f} min")
    
except FileNotFoundError:
    print("\n⚠ WARNING: Walking-only file not found!")
    print("You need to create 'library_travel_times_walk_only.csv' first.")
    raise

# ----------------------------------------------------------------------------
# PART 2: Convert census percentages to counts
# ----------------------------------------------------------------------------
print("\n" + "="*65)
print("STEP 2: Converting demographic percentages to counts")
print("="*65)

# Fill NaN values
for col in ['Unemployed, %', 'In core housing need, %', 'Immigrated within the last 10 years, %',
            'First generation immigrants, %', 'Visible minority, %', 'Low-income households (after-tax)',
            'Seniors, %', 'Children & Pre-teens, %']:
    if col in DA_Census.columns:
        DA_Census[col] = DA_Census[col].fillna(0)

# Unemployed
labour_force = DA_Census['Labour - Total Sex / Total - Population aged 15 years and over by labour force status - 25% sample data ; Both sexes / In the labour force ; Both sexes'].fillna(0)
DA_Census['Unemployed_count'] = (DA_Census['Unemployed, %'] / 100) * labour_force

# Core housing need
housing_total = DA_Census['Housing - Total Sex / Total - Owner and tenant households with household total income greater than zero and shelter-cost-to-income ratio less than 100%, in non-farm, non-reserve private dwellings - 25% sample data '].fillna(0)
DA_Census['Core_housing_need_count'] = (DA_Census['In core housing need, %'] / 100) * housing_total

# Recent immigrants
gen_status_total = DA_Census['Immigration - Total Sex / Total - Generation status for the population in private households - 25% sample data ; Both sexes '].fillna(0)
DA_Census['Recent_immigrant_count'] = (DA_Census['Immigrated within the last 10 years, %'] / 100) * gen_status_total

# First generation immigrants
immigrant_total = DA_Census['Immigration - Total Sex / Total - Immigrant status and period of immigration for the population in private households - 25% sample data ; Both sexes / Immigrants ; Both sexes'].fillna(0)
DA_Census['First_gen_immigrant_count'] = (DA_Census['First generation immigrants, %'] / 100) * immigrant_total

# Visible minority
vm_total = DA_Census['Indigenous and Visible Minorities - Total Sex / Total - Visible minority for the population in private households - 25% sample data ; Both sexes '].fillna(0)
DA_Census['Visible_minority_count'] = (DA_Census['Visible minority, %'] / 100) * vm_total

# Low income
income_total = DA_Census['Income - Total Sex / Total - After-tax income groups in 2020 for the population aged 15 years and over in private households - 100% data ; Both sexes'].fillna(0)

# Seniors and children
pop_total = DA_Census['Population and dwelling counts / Population, 2021'].fillna(0)
DA_Census['Seniors_count'] = (DA_Census['Seniors, %'] / 100) * pop_total
DA_Census['Children_count'] = (DA_Census['Children & Pre-teens, %'] / 100) * pop_total

print("✓ Conversion complete")

# ----------------------------------------------------------------------------
# PART 3: Spatial overlay
# ----------------------------------------------------------------------------
print("\n" + "="*65)
print("STEP 3: Performing spatial overlay")
print("="*65)

projected_crs = "EPSG:26917"

# Project layers
hex_projected = hex_with_times.to_crs(projected_crs)
hex_projected['hex_id'] = hex_projected['hex_id'].astype(str)  # Convert to string for overlay

DA_projected = DA_Census.to_crs(projected_crs)
DA_projected['da_id'] = DA_projected.index.astype(str)

# Prepare for overlay
hex_overlay = hex_projected[['hex_id', 'geometry']].copy()

demog_count_cols = ['Unemployed_count', 'Core_housing_need_count', 'Recent_immigrant_count',
                    'First_gen_immigrant_count', 'Visible_minority_count', 'Low_income_count',
                    'Seniors_count', 'Children_count']

da_overlay = DA_projected[['da_id', 'DAUID', 'geometry', 'Population and dwelling counts / Population, 2021', 
                           *demog_count_cols]].copy()

# Intersect
hex_with_da = gpd.overlay(hex_overlay, da_overlay, how='intersection')
print(f"Created {len(hex_with_da):,} intersection pieces")

# Calculate area weights
hex_with_da['intersection_area'] = hex_with_da.geometry.area
da_total_area = DA_projected.set_index('da_id')['geometry'].area
hex_with_da['da_area'] = hex_with_da['da_id'].map(da_total_area)
hex_with_da['weight'] = hex_with_da['intersection_area'] / hex_with_da['da_area']

# Allocate populations
all_count_cols = ['Population and dwelling counts / Population, 2021', *demog_count_cols]
for col in all_count_cols:
    hex_with_da[f'{col}_alloc'] = hex_with_da[col] * hex_with_da['weight']

# Aggregate to hexagons
hex_aggregated = hex_with_da.groupby('hex_id').agg({
    'geometry': 'first',
    **{f'{col}_alloc': 'sum' for col in all_count_cols}
}).reset_index()

hex_aggregated = gpd.GeoDataFrame(hex_aggregated, geometry='geometry', crs=projected_crs)

# Rename population columns
hex_aggregated.columns = ['hex_id', 'geometry',
                          'pop_total', 'unemployed', 'core_housing_need',
                          'recent_immigrants', 'first_gen_immigrants',
                          'visible_minority', 'low_income', 
                          'seniors', 'children']

# Convert hex_id back to int for merging with travel times
hex_aggregated['hex_id'] = hex_aggregated['hex_id'].astype(int)

# Add travel times back (CRITICAL FIX: ensure consistent types)
hex_with_times_int = hex_with_times.copy()
hex_with_times_int['hex_id'] = hex_with_times_int['hex_id'].astype(int)

hex_aggregated = hex_aggregated.merge(
    hex_with_times_int[['hex_id', 'walk_transit_minutes', 'walk_only_minutes']], 
    on='hex_id', 
    how='left'
)

print(f"Aggregated to {len(hex_aggregated):,} hexagons")

# ----------------------------------------------------------------------------
# PART 4: Calculate weighted travel times for BOTH modes
# ----------------------------------------------------------------------------
print("\n" + "="*65)
print("STEP 4: WEIGHTED AVERAGE TRAVEL TIME BY MODE (minutes)")
print("="*65)

demographic_labels = {
    'pop_total': 'Total Population',
    'unemployed': 'Unemployed',
    'core_housing_need': 'In Core Housing Need',
    'recent_immigrants': 'Recent Immigrants (last 10 yrs)',
    'first_gen_immigrants': 'First Generation Immigrants',
    'visible_minority': 'Visible Minority',
    'low_income': 'Low Income',
    'seniors': 'Seniors (65+)',
    'children': 'Children & Pre-teens (5-14)'
}

# Calculate for both modes
results = []
for col, label in demographic_labels.items():
    mask = hex_aggregated[col] > 0
    if mask.sum() > 0:
        # Walking + Transit
        wt_sum = (hex_aggregated.loc[mask, 'walk_transit_minutes'] * 
                  hex_aggregated.loc[mask, col]).sum()
        total_pop = hex_aggregated.loc[mask, col].sum()
        wt_avg = wt_sum / total_pop if total_pop > 0 else np.nan
        
        # Walking Only
        w_sum = (hex_aggregated.loc[mask, 'walk_only_minutes'] * 
                 hex_aggregated.loc[mask, col]).sum()
        w_avg = w_sum / total_pop if total_pop > 0 else np.nan
        
        results.append({
            'Demographic Group': label, 
            'Walking + Transit': round(wt_avg, 2),
            'Walking Only': round(w_avg, 2),
            'Transit Benefit': round(w_avg - wt_avg, 2)
        })

weighted_comparison = pd.DataFrame(results)

# Display as pivot table (matching Skating Rinks format)
pivot_table = weighted_comparison.set_index('Demographic Group')[['Walking + Transit', 'Walking Only']]
print("\n" + pivot_table.to_string())

# Display transit benefit
print("\n" + "-"*65)
print("TRANSIT BENEFIT (minutes saved by using transit)")
print("-"*65)
benefit_df = weighted_comparison[['Demographic Group', 'Transit Benefit']]
print(benefit_df.to_string(index=False))

# ----------------------------------------------------------------------------
# PART 5: Population distribution by travel time category
# ----------------------------------------------------------------------------
print("\n" + "="*65)
print("STEP 5: POPULATION DISTRIBUTION BY TRAVEL TIME CATEGORY")
print("="*65)

def categorize_time(minutes):
    if minutes < 15: return "Under 15 Mins"
    elif minutes < 30: return "15-30 Mins"
    elif minutes < 45: return "31-45 Mins"
    elif minutes <= 60: return "46-60 Mins"
    else: return "More than 60 Mins"

category_order = ['Under 15 Mins', '15-30 Mins', '31-45 Mins', '46-60 Mins', 'More than 60 Mins']

# For Walking + Transit
hex_aggregated['wt_category'] = hex_aggregated['walk_transit_minutes'].apply(categorize_time)
wt_summary = hex_aggregated.groupby('wt_category')[list(demographic_labels.keys())].sum()
wt_summary = wt_summary.reindex(category_order)
wt_pct = wt_summary.div(wt_summary.sum()) * 100

# For Walking Only
hex_aggregated['w_category'] = hex_aggregated['walk_only_minutes'].apply(categorize_time)
w_summary = hex_aggregated.groupby('w_category')[list(demographic_labels.keys())].sum()
w_summary = w_summary.reindex(category_order)
w_pct = w_summary.div(w_summary.sum()) * 100

print("\n--- WALKING + TRANSIT ---")
wt_output = wt_pct.T
wt_output.columns = category_order
wt_output.index = [demographic_labels[col] for col in wt_output.index]
print(wt_output.round(1))

print("\n--- WALKING ONLY ---")
w_output = w_pct.T
w_output.columns = category_order
w_output.index = [demographic_labels[col] for col in w_output.index]
print(w_output.round(1))

# ----------------------------------------------------------------------------
# PART 6: Save results
# ----------------------------------------------------------------------------
print("\n" + "="*65)
print("STEP 6: Saving results")
print("="*65)

weighted_comparison.to_csv("data/library_access_mode_comparison.csv", index=False)
wt_pct.to_csv("data/library_access_walk_transit_distribution.csv")
w_pct.to_csv("data/library_access_walk_only_distribution.csv")
hex_aggregated.to_file("data/hexagon_library_access_both_modes.geojson", driver='GeoJSON')

print("\n✓ Results saved successfully!")
print("  - data/library_access_mode_comparison.csv")
print("  - data/library_access_walk_transit_distribution.csv")
print("  - data/library_access_walk_only_distribution.csv")
print("  - data/hexagon_library_access_both_modes.geojson")

# ----------------------------------------------------------------------------
# SUMMARY
# ----------------------------------------------------------------------------
print("\n" + "="*65)
print("FINAL SUMMARY")
print("="*65)
total_pop = hex_aggregated['pop_total'].sum()
print(f"Total population analyzed: {total_pop:,.0f}")
print(f"\nAverage travel times for Total Population:")
wt = weighted_comparison[weighted_comparison['Demographic Group'] == 'Total Population']['Walking + Transit'].values[0]
w = weighted_comparison[weighted_comparison['Demographic Group'] == 'Total Population']['Walking Only'].values[0]
print(f"  Walking + Transit: {wt:.1f} min")
print(f"  Walking Only:      {w:.1f} min")
print(f"  Transit saves:     {w - wt:.1f} min")

## Export Static Choropleth Maps as PNG

Generates one PNG per demographic variable. Copy outputs to `static/maps/` in the repo.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import geopandas as gpd
import numpy as np
from matplotlib.patches import Patch

ct = gpd.read_file('census_tracts_demographics.geojson')

# Rotate to match bearing -17 (rotate geometry by +17 degrees)
# Project to UTM, rotate, then plot
ct_proj = ct.to_crs('EPSG:26917')

# Get centroid for rotation
cx = ct_proj.geometry.centroid.x.mean()
cy = ct_proj.geometry.centroid.y.mean()

import shapely.affinity

def rotate_gdf(gdf, angle_deg, cx, cy):
    gdf = gdf.copy()
    gdf['geometry'] = gdf.geometry.apply(
        lambda g: shapely.affinity.rotate(g, angle_deg, origin=(cx, cy))
    )
    return gdf

ct_rotated = rotate_gdf(ct_proj, -17, cx, cy)

variables = [
    {
        'col': 'visible_minority_pct',
        'title': 'Visible Minority',
        'colors': ['#F0E9F1','#C4A7C9','#9865A1','#6D247A'],
        'breaks': [31.4, 52.1, 74.4],
    },
    {
        'col': 'low_income_pct',
        'title': 'Low Income',
        'colors': ['#FBECEA','#F1B5AD','#E67D70','#DC4633'],
        'breaks': [28.0, 34.1, 37.8],
    },
    {
        'col': 'recent_immigrants_pct',
        'title': 'Recent Immigrants (last 10 yrs)',
        'colors': ['#E5F2F5','#99CBDA','#4CA5BE','#007FA3'],
        'breaks': [3.2, 5.6, 8.3],
    },
    {
        'col': 'first_gen_immigrants_pct',
        'title': 'First Generation Immigrants',
        'colors': ['#E5F5F3','#99D9CF','#4CBDAC','#00A189'],
        'breaks': [37.6, 52.9, 63.3],
    },
    {
        'col': 'seniors_pct',
        'title': 'Seniors (65+)',
        'colors': ['#E8EBEF','#A5AFC1','#617393','#1E3765'],
        'breaks': [14.1, 17.0, 20.4],
    },
    {
        'col': 'children_pct',
        'title': 'Children (0–14)',
        'colors': ['#F3F8EA','#D1E5AB','#AFD26C','#8DBF2E'],
        'breaks': [11.7, 14.1, 16.3],
    },
]

for v in variables:
    colors = v['colors']
    breaks = [0] + v['breaks'] + [100]

    def get_color(val):
        for i in range(len(breaks) - 1):
            if val <= breaks[i + 1]:
                return colors[min(i, len(colors) - 1)]
        return colors[-1]

    ct_rotated['_color'] = ct[v['col']].apply(get_color)

    fig, ax = plt.subplots(1, 1, figsize=(12, 7))
    fig.patch.set_facecolor('white')
    ax.set_facecolor('white')

    ct_rotated.plot(
        ax=ax,
        color=ct_rotated['_color'],
        edgecolor='white',
        linewidth=0.3
    )

    ax.axis('off')
    ax.set_aspect('equal')

    # Remove all margins
    ax.margins(0)
    plt.tight_layout(pad=0)

    # Legend — bottom left
    labels = [f'< {v["breaks"][0]}%'] + \
             [f'{v["breaks"][i]}–{v["breaks"][i+1]}%'
              for i in range(len(v["breaks"]) - 1)] + \
             [f'> {v["breaks"][-1]}%']

    patches = [Patch(facecolor=c, edgecolor='white', label=l)
               for c, l in zip(colors, labels)]

    legend = ax.legend(
        handles=patches,
        loc='lower right',
        fontsize=9,
        title=v['title'],
        title_fontsize=10,
        framealpha=1.0,
        edgecolor='#cccccc',
        borderpad=0.8,
        handlelength=1.5,
        handleheight=1.5,
    )
    legend.get_title().set_fontweight('bold')

    fname = f'static_map_{v["col"]}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight',
                facecolor='white', edgecolor='none')
    plt.close()
    print(f"Saved {fname}")

print("\nAll done! Copy to static/maps/ in your repo.")